Step 1: Loading a dataset, and cleaning them

In [1]:
import numpy as np
import pandas as pd

# Loading a dataset
data = pd.read_csv("synthetic_heart_disease_dataset.csv")

# important : Gender column removed because its not important feature to model, it can reduce accuracy
data = data.drop("Gender",axis=1)

# spliting the data into two (features/input and target/ouput)
x = data.iloc[:, :19]
y = data.iloc[:, 19]

# just cross verifying
# print(x.head)
# print(y.head)
# print(x.shape)
# print(y.shape)

Step 2: Converting text values to numerical value using LabelEncoder

In [2]:
from sklearn.preprocessing import LabelEncoder
encoders={}
text_values_columns = ["Smoking","Alcohol_Intake","Physical_Activity","Diet","Stress_Level"]

for col in text_values_columns:
    le = LabelEncoder() 
    x[col] = le.fit_transform(x[col])
    encoders[col] = le

# just cross verifying
# print(x.head)

Step 3: Spliting the dataset into two parts (train and test dataset)

In [3]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# just cross verifying
# print(len(x_train),len(y_train)) #train
# print(len(x_test),len(y_test)) #test

Step 4: Normalizing the values using StandardScaler

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_scaled = scaler.fit_transform(x_train) # learns and applies
x_test = scaler.transform(x_test) # it make changes from what it learn

Step 5: Training the model

In [5]:
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix
from sklearn.linear_model import LogisticRegression
import joblib

# The model use LogisticRegression Algo to train the data 
model=LogisticRegression(max_iter=1000) 

# training the data
model.fit(x_scaled,y_train)

#storing the trained model in a pickle file
joblib.dump({
    "model":model,
    "scaler":scaler,
    "encoders":encoders
    },
    "heart_disease_dataset.pkl"
)

['heart_disease_dataset.pkl']

Step 6:Checking the accuracy of the model

In [6]:
# predicting the output on untrained data
y_pred = model.predict(x_test)

# checking model accuracy
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.9263
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      5342
           1       0.92      0.92      0.92      4658

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.93     10000
weighted avg       0.93      0.93      0.93     10000

[[4979  363]
 [ 374 4284]]


In [ ]:
import joblib

def predict(new_data):
    saved = joblib.load("heart_disease_dataset.pkl")

    model = saved["model"]
    scaler = saved["scaler"]
    encoders = saved["encoders"]
    for col in encoders:
        print(col, "=>", encoders[col].classes_)
    new_data = new_data.drop("Gender",axis=1)

    for col in text_values_columns:
        new_data[col] = encoders[col].transform(new_data[col])
        print("Training classes:", encoders[col].classes_)
        print("New value:", new_data[col].iloc[0])

    new_data_scaled = scaler.transform(new_data)

    prediction = model.predict(new_data_scaled)

    return prediction

In [8]:
new_data = pd.DataFrame([{
     "Age": 42,
    "Gender": "Male",
    "Weight": 82,
    "Height": 175,
    "BMI": 26.8,
    "Smoking": "Never",
    "Alcohol_Intake": "None",
    "Physical_Activity": "Active",
    "Diet": "Healthy",
    "Stress_Level": "Low",
    "Hypertension": 0,
    "Diabetes": 0,
    "Hyperlipidemia": 0,
    "Family_History": 0,
    "Previous_Heart_Attack": 0,
    "Systolic_BP": 118,
    "Diastolic_BP": 76,
    "Heart_Rate": 72,
    "Blood_Sugar_Fasting": 91,
    "Cholesterol_Total": 185
}])


prediction = predict(new_data)

print("Prediction : ",prediction)

Training classes: ['Current' 'Former' 'Never']
New value: 2


ValueError: y contains previously unseen labels: 'None'